# Train Tokenizer **mix-dataset** 50.265 Vocab

# Import Library

In [1]:
from tokenizers import ByteLevelBPETokenizer

# Initialize a Byte-Pair Encoding(BPE) tokenizer
tokenizer = ByteLevelBPETokenizer()

print(tokenizer)

Tokenizer(vocabulary_size=0, model=ByteLevelBPE, add_prefix_space=False, lowercase=False, dropout=None, unicode_normalizer=None, continuing_subword_prefix=None, end_of_word_suffix=None, trim_offsets=False)


# Load Corpus

In [2]:
training_dataset = ('/kaggle/input/korpus-mix/mix_dataset.txt')

print(training_dataset)

/kaggle/input/korpus-mix/mix_dataset.txt


# Train Tokenizer

In [3]:
# Latih tokenizer

tokenizer.train(
    files=training_dataset, # data corpus
    vocab_size=50265,  # Ukuran total vocab asli BART 0-50264 total 50265 token
    min_frequency=2,   # Abaikan token yang muncul kurang dari 2 kali
    special_tokens=["<s>", "<pad>", "</s>", "<unk>", "<mask>"]  # Special token BART
)

In [4]:
print(f"Tokenizer setelah train: \n{tokenizer}")

Tokenizer setelah train: 
Tokenizer(vocabulary_size=50265, model=ByteLevelBPE, add_prefix_space=False, lowercase=False, dropout=None, unicode_normalizer=None, continuing_subword_prefix=None, end_of_word_suffix=None, trim_offsets=False)


In [5]:
import os

# Save the tokenizer
output_dir = "./mix-tokenizer"
os.makedirs(output_dir, exist_ok=True)
tokenizer.save_model(output_dir)

['./mix-tokenizer/vocab.json', './mix-tokenizer/merges.txt']

# Swap mask token index

Menyesuaikan index <mask> token, karena <mask> pada BART berda di index paing terakhir

In [6]:
import json

vocab_file = '/kaggle/working/mix-tokenizer/vocab.json'

# Baca vocab.json
with open(vocab_file, "r", encoding="utf-8") as f:
    vocab = json.load(f)

print("Ukuran vocabulary awal:", len(vocab))
print("Indeks <mask> awal:", vocab.get("<mask>"))

Ukuran vocabulary awal: 50265
Indeks <mask> awal: 4


In [7]:
# Check last index token
token_last_index = None
for token, idx in vocab.items():
    if idx == 50264:
        token_last_index = token
        break
print(f"Token di indeks terakhir: {token_last_index}")

Token di indeks terakhir: ĠAnonim


In [8]:
# Tukar indeks: Pindahkan token di indeks terakhir ke 4
if token_last_index:
    vocab[token_last_index] = 4  # Pindahkan token lama ke indeks 4
else:
    print("Tidak ada token di indeks 50264")

# Tambahkan <mask> di indeks 50264
vocab["<mask>"] = 50264

In [9]:
# Check Index <mask>
print("Ukuran vocabulary setelah modifikasi:", len(vocab))
print("Indeks <mask> setelah modifikasi:", vocab.get("<mask>"))
print(f"Indeks {token_last_index} setelah modifikasi:", vocab.get(token_last_index))

Ukuran vocabulary setelah modifikasi: 50265
Indeks <mask> setelah modifikasi: 50264
Indeks ĠAnonim setelah modifikasi: 4


In [10]:
# Periksa duplikasi <mask>
mask_count = sum(1 for token in vocab if token == "<mask>")
print(f"Jumlah <mask> di vocabulary: {mask_count}")

Jumlah <mask> di vocabulary: 1


In [11]:
# Sort vocab secara asc
sorted_vocab = dict(sorted(vocab.items(), key=lambda x: x[1]))

## Simpan vocab.json

In [12]:
# simpan vocab.json

# Direktori
new_output_dir = '/kaggle/working/mix-tokenizer-new'
os.makedirs(new_output_dir, exist_ok=True)

# Simpan output vocab.json
output_json_file = os.path.join(new_output_dir, "vocab.json")
with open(output_json_file, "w", encoding="utf-8") as f:
    json.dump(sorted_vocab, f, ensure_ascii=False, indent=2)
print(f"Output disimpan ke: {output_json_file}")

Output disimpan ke: /kaggle/working/mix-tokenizer-new/vocab.json


## Simpan merge.txt

In [13]:
import shutil

# Buat copy merge.txt dan simpan ke folder baru

merges_file = '/kaggle/working/mix-tokenizer/merges.txt'

merges_copy = '/kaggle/working/mix-tokenizer-new'

# Buat salinan merges.txt ke direktori baru
merges_copy_file = os.path.join(merges_copy, "merges.txt")

shutil.copy(merges_file, merges_copy_file)

print(f"Salinan merges.txt dibuat di: {merges_copy_file}")

Salinan merges.txt dibuat di: /kaggle/working/mix-tokenizer-new/merges.txt


In [14]:
print(os.listdir('/kaggle/working/mix-tokenizer-new'))

['vocab.json', 'merges.txt']


# Upload Tokenizer ke Kagglehub

In [16]:
import kagglehub

kagglehub.login()

# Tokenizer custom path
MY_TOKENIZER_DIR = '/kaggle/working/mix-tokenizer-new'

# Tokenizer name
TOKENIZER_SLUG = 'tokenizer-mix' # Nama tokenizer
VARIATION_SLUG = 'tokenizer-mix-50265' # Variasi tokenizer

# Upload model ke kagglehub
kagglehub.model_upload(
  handle = f"jawawahirul/{TOKENIZER_SLUG}/transformers/{VARIATION_SLUG}",
  local_model_dir = MY_TOKENIZER_DIR,
  version_notes = 'Update 2025-05-29'
)

Uploading Model https://www.kaggle.com/models/jawawahirul/tokenizer-mix/transformers/tokenizer-mix-50265 ...
Starting upload for file /kaggle/working/mix-tokenizer-new/vocab.json


Uploading: 100%|██████████| 1.01M/1.01M [00:00<00:00, 4.73MB/s]

Upload successful: /kaggle/working/mix-tokenizer-new/vocab.json (991KB)
Starting upload for file /kaggle/working/mix-tokenizer-new/merges.txt



Uploading: 100%|██████████| 472k/472k [00:00<00:00, 2.11MB/s]

Upload successful: /kaggle/working/mix-tokenizer-new/merges.txt (461KB)


Your model instance has been created.
Files are being processed...
See at: https://www.kaggle.com/models/jawawahirul/tokenizer-mix/transformers/tokenizer-mix-50265
